In [1]:
print("Hello World")

Hello World


In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Installation Unsloth qui gère automatiquement toutes les dépendances
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

print("✅ Installation terminée")
print("⚠️  Runtime → Redémarrer le runtime maintenant")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 118.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.6/869.6 kB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 86.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 20.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 28.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 24.6 MB/s eta 0:00:00
✅ Installation termin

In [13]:
# ══════════════════════════════════════════════════════════════
# CELLULE 2 — VÉRIFICATION ENVIRONNEMENT
# Après installation Unsloth, vérifier le GPU et les versions
# ══════════════════════════════════════════════════════════════
import torch
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print(f"✅ GPU  : {torch.cuda.get_device_name(0)}")
print(f"✅ VRAM : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GiB")
print(f"✅ CUDA : {torch.version.cuda}")
print(f"✅ PyTorch : {torch.__version__}")

✅ GPU  : Tesla T4
✅ VRAM : 14.6 GiB
✅ CUDA : 12.8
✅ PyTorch : 2.10.0+cu128


In [14]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Chemins ──────────────────────────────────────────────────
BASE          = "/content/drive/MyDrive/Lynes"
CHEMIN_TRAIN  = f"{BASE}/dataset/train.jsonl"
CHEMIN_EVAL   = f"{BASE}/dataset/eval.jsonl"
DOSSIER_LORA  = f"{BASE}/finetuned_lora_v2"
DOSSIER_MERGE = f"{BASE}/model_final_v2"
MODEL_NAME    = "Qwen/Qwen2.5-3B-Instruct"

os.makedirs(DOSSIER_LORA,  exist_ok=True)
os.makedirs(DOSSIER_MERGE, exist_ok=True)

# ── Paramètres LoRA ──────────────────────────────────────────
LORA_R       = 32
LORA_ALPHA   = 64
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj", "v_proj", "k_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj"
]

# ── Paramètres entraînement ──────────────────────────────────
NUM_EPOCHS     = 5
BATCH_SIZE     = 1
GRAD_ACCUM     = 8
LEARNING_RATE  = 1e-4
MAX_SEQ_LEN    = 768
WARMUP_RATIO   = 0.05
WEIGHT_DECAY   = 0.01

print("✅ Configuration OK")
print(f"   Train  : {CHEMIN_TRAIN}")
print(f"   Eval   : {CHEMIN_EVAL}")
print(f"   LoRA   : {DOSSIER_LORA}")
print(f"   Merge  : {DOSSIER_MERGE}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Configuration OK
   Train  : /content/drive/MyDrive/Lynes/dataset/train.jsonl
   Eval   : /content/drive/MyDrive/Lynes/dataset/eval.jsonl
   LoRA   : /content/drive/MyDrive/Lynes/finetuned_lora_v2
   Merge  : /content/drive/MyDrive/Lynes/model_final_v2


In [12]:
# ══════════════════════════════════════════════════════════════
# CELLULE 4 — VÉRIFICATION DU DATASET
# ══════════════════════════════════════════════════════════════
import json, os

for nom, chemin in [("train", CHEMIN_TRAIN), ("eval", CHEMIN_EVAL)]:
    if os.path.exists(chemin):
        with open(chemin) as f:
            nb = sum(1 for l in f if l.strip())
        taille = os.path.getsize(chemin) / 1024
        print(f"✅ {nom}.jsonl : {nb} exemples ({taille:.0f} KB)")
    else:
        print(f"❌ Introuvable : {chemin}")

print("\n🔍 Exemple train :")
with open(CHEMIN_TRAIN) as f:
    ex = json.loads(f.readline())
for msg in ex["messages"]:
    role = msg["role"].upper()
    if role == "SYSTEM":
        print(f"\n[SYSTEM] → {msg['content'][:80]}...")
    else:
        print(f"\n[{role}]\n{msg['content'][:150]}")

❌ Introuvable : /content/drive/MyDrive/Lynes/dataset/train.jsonl
❌ Introuvable : /content/drive/MyDrive/Lynes/dataset/eval.jsonl

🔍 Exemple train :


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Lynes/dataset/train.jsonl'

In [15]:
# ══════════════════════════════════════════════════════════════
# CELLULE 5 — CHARGEMENT MODÈLE + TOKENIZER AVEC UNSLOTH
# Unsloth charge le modèle en 4-bit sans bitsandbytes
# → fonctionne parfaitement sur T4 CUDA 12.x
# FastLanguageModel.from_pretrained charge les deux en même temps
# ══════════════════════════════════════════════════════════════
from unsloth import FastLanguageModel
import torch, gc

gc.collect()
torch.cuda.empty_cache()

print("⏳ Chargement modèle 4-bit avec Unsloth...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,          # None = détection automatique (bfloat16 sur T4)
    load_in_4bit=True,   # Quantization 4-bit sans bitsandbytes
)

vram_used  = (torch.cuda.get_device_properties(0).total_memory
              - torch.cuda.mem_get_info()[0]) / 1024**3
vram_libre = torch.cuda.mem_get_info()[0] / 1024**3

print(f"\n✅ Modèle chargé en 4-bit")
print(f"   VRAM utilisée  : {vram_used:.1f} GiB")
print(f"   VRAM restante  : {vram_libre:.1f} GiB")
# Attendu : ~4-5 GiB utilisés, ~10-11 GiB restants

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
⏳ Chargement modèle 4-bit avec Unsloth...
==((====))==  Unsloth 2026.5.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.

✅ Modèle chargé en 4-bit
   VRAM utilisée  : 2.4 GiB
   VRAM restante  : 12.2 GiB


In [16]:
# ══════════════════════════════════════════════════════════════
# CELLULE 6 — APPLICATION LORA AVEC UNSLOTH
# get_peft_model d'Unsloth est optimisé pour la vitesse
# use_gradient_checkpointing="unsloth" = version optimisée
# qui économise encore plus de VRAM que la version standard
# ══════════════════════════════════════════════════════════════

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    use_gradient_checkpointing="unsloth",  # ← version optimisée Unsloth
    random_state=42,
    use_rslora=False,
)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ LoRA appliqué")
print(f"   Paramètres totaux       : {total:,}")
print(f"   Paramètres entraînables : {trainable:,}")
print(f"   Pourcentage entraîné    : {100*trainable/total:.3f}%")

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.5.7 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


✅ LoRA appliqué
   Paramètres totaux       : 1,859,989,504
   Paramètres entraînables : 59,867,136
   Pourcentage entraîné    : 3.219%


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 7 — PRÉPARATION DATASET POUR UNSLOTH
# Unsloth gère le masquage lui-même via train_on_responses_only
# On fournit juste le texte formaté avec le chat template
# ══════════════════════════════════════════════════════════════
import json
from datasets import Dataset

def charger_jsonl(chemin):
    exemples = []
    with open(chemin, "r", encoding="utf-8") as f:
        for ligne in f:
            ligne = ligne.strip()
            if ligne:
                exemples.append(json.loads(ligne))
    return exemples

def formater_exemple(exemple):
    """Applique le chat template Qwen2.5 et retourne le texte."""
    texte = tokenizer.apply_chat_template(
        exemple["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": texte}

print("⏳ Chargement dataset...")
train_raw = charger_jsonl(CHEMIN_TRAIN)
eval_raw  = charger_jsonl(CHEMIN_EVAL)
print(f"   Train : {len(train_raw)} | Eval : {len(eval_raw)}")

train_ds = Dataset.from_list(train_raw).map(formater_exemple)
eval_ds  = Dataset.from_list(eval_raw).map(formater_exemple)

# Garder uniquement la colonne text
train_ds = train_ds.select_columns(["text"])
eval_ds  = eval_ds.select_columns(["text"])

print(f"\n✅ Dataset prêt")
print(f"\n🔍 Exemple formaté :")
print(train_ds[0]["text"][:300])

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 8 — CONFIGURATION LORA + TRAINER
# Garde exactement ce qui marche, change UNIQUEMENT optim
# ══════════════════════════════════════════════════════════════
from trl import SFTConfig, SFTTrainer
from unsloth import is_bfloat16_supported

steps_par_epoch = len(train_ds) // (BATCH_SIZE * GRAD_ACCUM)
total_steps     = steps_par_epoch * NUM_EPOCHS
warmup_steps    = max(1, int(total_steps * WARMUP_RATIO))

sft_config = SFTConfig(
    output_dir=DOSSIER_LORA,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,

    # ← SEUL CHANGEMENT : adamw_torch au lieu de paged_adamw_8bit
    # paged_adamw_8bit → bitsandbytes → libnvJitLink.so.13 manquant → crash
    # adamw_torch      → PyTorch pur  → aucune dépendance bitsandbytes
    optim="adamw_torch",

    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",
    bf16=True if is_bfloat16_supported() else False,
    fp16=False if is_bfloat16_supported() else True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=0,
    max_grad_norm=1.0,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=sft_config,
    train_on_responses_only=True,
)
print("✅ Trainer configuré")
print(f"   Steps/epoch   : {steps_par_epoch}")
print(f"   Total steps   : {total_steps}")
print(f"   Durée estimée : ~{total_steps*4//60} à {total_steps*7//60} minutes")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 9 — ENTRAÎNEMENT
# ══════════════════════════════════════════════════════════════

print("\n🚀 LANCEMENT ENTRAÎNEMENT")
print("=" * 50)
trainer.train()
print("✅ ENTRAÎNEMENT TERMINÉ")

model.save_pretrained(DOSSIER_LORA)
tokenizer.save_pretrained(DOSSIER_LORA)
print(f"✅ Adaptateurs LoRA sauvegardés : {DOSSIER_LORA}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 10 — SAUVEGARDE DES ADAPTATEURS LORA
# ══════════════════════════════════════════════════════════════

model.save_pretrained(DOSSIER_LORA)
tokenizer.save_pretrained(DOSSIER_LORA)

print(f"✅ Adaptateurs LoRA sauvegardés : {DOSSIER_LORA}")
for f in sorted(os.listdir(DOSSIER_LORA)):
    chemin = os.path.join(DOSSIER_LORA, f)
    if os.path.isfile(chemin):
        taille = os.path.getsize(chemin) / 1024**2
        print(f"   {f:<45} {taille:.1f} MB")


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 11 — MERGE LORA + MODÈLE DE BASE
# On libère la VRAM avant de recharger le modèle de base
# ══════════════════════════════════════════════════════════════

import gc, torch, os
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# Libérer la VRAM
print("🧹 Libération VRAM...")
del model
del trainer
gc.collect()
torch.cuda.empty_cache()
print(f"   VRAM libre : {torch.cuda.mem_get_info()[0]/1024**3:.1f} GiB")

# Recharger le modèle de base en float16
print("⏳ Chargement modèle de base (float16)...")
model_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# Merger les adaptateurs
print("⏳ Merge des adaptateurs LoRA...")
model_merge = PeftModel.from_pretrained(model_base, DOSSIER_LORA)
model_merge = model_merge.merge_and_unload()

# Déplacer sur CPU pour la sauvegarde
model_merge = model_merge.to("cpu")

# Sauvegarder
print("⏳ Sauvegarde du modèle final...")
os.makedirs(DOSSIER_MERGE, exist_ok=True)
model_merge.save_pretrained(
    DOSSIER_MERGE,
    max_shard_size="2GB",
    safe_serialization=True,
)
tokenizer_merge = AutoTokenizer.from_pretrained(DOSSIER_LORA, trust_remote_code=True)
tokenizer_merge.save_pretrained(DOSSIER_MERGE)

del model_merge, model_base
gc.collect()
torch.cuda.empty_cache()

print(f"\n✅ Modèle final sauvegardé : {DOSSIER_MERGE}")
for f in sorted(os.listdir(DOSSIER_MERGE)):
    chemin = os.path.join(DOSSIER_MERGE, f)
    if os.path.isfile(chemin):
        taille = os.path.getsize(chemin) / 1024**2
        print(f"   {f:<45} {taille:.1f} MB")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 12 — CHARGEMENT MODÈLE POUR INFÉRENCE
# ══════════════════════════════════════════════════════════════

import gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

gc.collect()
torch.cuda.empty_cache()

# Option A : modèle mergé (recommandé)
if os.path.exists(DOSSIER_MERGE):
    print("⏳ Chargement depuis model_final (merge)...")
    infer_tokenizer = AutoTokenizer.from_pretrained(
        DOSSIER_MERGE, trust_remote_code=True
    )
    infer_tokenizer.pad_token    = infer_tokenizer.eos_token
    infer_tokenizer.padding_side = "right"

    infer_model = AutoModelForCausalLM.from_pretrained(
        DOSSIER_MERGE,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    print("✅ Modèle mergé chargé")

# Option B : base + LoRA (si merge absent)
else:
    print("⏳ Merge absent → chargement base + LoRA...")
    infer_tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME, trust_remote_code=True
    )
    infer_tokenizer.pad_token    = infer_tokenizer.eos_token
    infer_tokenizer.padding_side = "right"

    quant2 = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quant2,
        device_map="auto",
        trust_remote_code=True,
    )
    infer_model = PeftModel.from_pretrained(base, DOSSIER_LORA)
    print("✅ Base + LoRA chargé")

infer_model.eval()
print(f"   VRAM : {torch.cuda.memory_allocated()/1024**3:.1f} GiB utilisés")


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 12 — CONFIGURATION DE L'ENTRAÎNEMENT
# ══════════════════════════════════════════════════════════════
from trl import SFTConfig, SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
import os

os.makedirs(DOSSIER_LORA, exist_ok=True)

# Calculer les steps
steps_par_epoch = len(train_tok) // (BATCH_SIZE * GRAD_ACCUM)
total_steps     = steps_par_epoch * NUM_EPOCHS
warmup_steps    = max(1, int(total_steps * WARMUP_RATIO))
eval_steps      = max(10, steps_par_epoch // 2)

print(f"   Steps par époque : {steps_par_epoch}")
print(f"   Total steps      : {total_steps}")
print(f"   Warmup steps     : {warmup_steps}")
print(f"   Eval steps       : {eval_steps}")

training_args = TrainingArguments(
    output_dir=DOSSIER_LORA,

    # ── Durée ────────────────────────────────────────────────
    num_train_epochs=NUM_EPOCHS,

    # ── Batch ────────────────────────────────────────────────
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    # ── Optimiseur ───────────────────────────────────────────
    # paged_adamw_8bit : gère les états optimiseur en mémoire paginée
    # → moins de VRAM que AdamW standard
    optim="paged_adamw_8bit",
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",      # LR descend en cosinus = stable

    # ── Précision ────────────────────────────────────────────
    fp16=True,
    bf16=False,

    # ── Gradient checkpointing ───────────────────────────────
    # Réduit la VRAM en recalculant les activations au lieu de les stocker
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # ── Évaluation et sauvegarde ─────────────────────────────
    eval_strategy="steps",
    eval_steps=eval_steps,
    save_strategy="steps",
    save_steps=eval_steps,
    save_total_limit=2,              # Garder les 2 meilleurs checkpoints
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # ── Logging ──────────────────────────────────────────────
    logging_steps=10,
    logging_dir=f"{DOSSIER_LORA}/logs",
    report_to="none",

    # ── Divers ───────────────────────────────────────────────
    dataloader_num_workers=0,
    group_by_length=True,            # Regroupe les séquences de même longueur
    max_grad_norm=1.0,               # Gradient clipping
    label_names=["labels"],
)

# Data collator : gère le padding dynamique dans le batch
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8,
    label_pad_token_id=-100,
)

# Trainer
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=data_collator,
    args=training_args,
)

print("\n✅ Trainer configuré")
print(f"   Total steps     : {total_steps}")
print(f"   Durée estimée   : ~{total_steps * 5 // 60} à {total_steps * 8 // 60} minutes")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 8 — LANCEMENT DE L'ENTRAÎNEMENT
# ══════════════════════════════════════════════════════════════
print("🚀 LANCEMENT DE L'ENTRAÎNEMENT")
print("=" * 60)
print("   Loss affichée toutes les 10 steps")
print(f"  Validation toutes les {eval_steps} steps")
print()
print("   Objectif de convergence :")
print("   Step 50  → loss ~1.5")
print("   Step 150 → loss ~0.8")
print("   Step 300 → loss ~0.4")
print("   Step 500 → loss ~0.2")
print("   Val loss stable ≈ train loss → pas d'overfitting\n")

trainer.train()

print("\n✅ ENTRAÎNEMENT TERMINÉ")

# Afficher la courbe de loss
logs = trainer.state.log_history
train_logs = [l for l in logs if "loss" in l and "eval_loss" not in l]
eval_logs  = [l for l in logs if "eval_loss" in l]

print(f"\n   Loss finale train : {train_logs[-1]['loss']:.4f}")
if eval_logs:
    print(f"   Loss finale eval  : {eval_logs[-1]['eval_loss']:.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 9 — SAUVEGARDE DES ADAPTATEURS LORA
# ══════════════════════════════════════════════════════════════
import os

model.save_pretrained(DOSSIER_LORA)
tokenizer.save_pretrained(DOSSIER_LORA)

print(f"✅ Adaptateurs LoRA sauvegardés : {DOSSIER_LORA}")
print(f"\n   Fichiers créés :")
for f in sorted(os.listdir(DOSSIER_LORA)):
    chemin = os.path.join(DOSSIER_LORA, f)
    if os.path.isfile(chemin):
        taille = os.path.getsize(chemin) / 1024**2
        print(f"   {f:<45} {taille:.1f} MB")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 10 — MERGE LORA + MODÈLE DE BASE
# ══════════════════════════════════════════════════════════════
import gc, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

os.makedirs(DOSSIER_MERGE, exist_ok=True)

# Libérer la VRAM avant de recharger
print("🧹 Libération VRAM...")
del model
del trainer
gc.collect()
torch.cuda.empty_cache()
print(f"   VRAM libre : {torch.cuda.mem_get_info()[0]/1024**3:.1f} GiB")

# Recharger le modèle de base en float16
print("\n⏳ Chargement modèle de base (float16)...")
model_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# Merger les adaptateurs LoRA
print("⏳ Merge des adaptateurs LoRA...")
model_merge = PeftModel.from_pretrained(model_base, DOSSIER_LORA)
model_merge = model_merge.merge_and_unload()

# Sauvegarder
print("⏳ Sauvegarde du modèle final...")
model_merge.save_pretrained(DOSSIER_MERGE, max_shard_size="2GB", safe_serialization=True)
tokenizer.save_pretrained(DOSSIER_MERGE)

# Nettoyer
del model_base, model_merge
gc.collect()
torch.cuda.empty_cache()

print(f"\n✅ Modèle final sauvegardé : {DOSSIER_MERGE}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE 11 — TEST DU MODÈLE FINAL
# ══════════════════════════════════════════════════════════════
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

SYSTEM_PROMPT_TEST = open(CHEMIN_TRAIN).readline()
SYSTEM_PROMPT_TEST = json.loads(SYSTEM_PROMPT_TEST)["messages"][0]["content"]

infer_tokenizer = AutoTokenizer.from_pretrained(DOSSIER_MERGE, trust_remote_code=True)
infer_model     = AutoModelForCausalLM.from_pretrained(
    DOSSIER_MERGE,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
infer_model.eval()

def predire(commentaire, owner="", topology="", vendors=""):
    """Prédit la cause d'un incident."""
    user = f"Commentaire : \"{commentaire}\""
    if owner:    user += f"\nOwner : {owner}"
    if topology: user += f"\nTopology : {topology}"
    if vendors:  user += f"\nVendors : {vendors}"

    messages = [
        {"role": "system",  "content": SYSTEM_PROMPT_TEST},
        {"role": "user",    "content": user},
    ]
    prompt = infer_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = infer_tokenizer(prompt, return_tensors="pt").to(infer_model.device)

    with torch.no_grad():
        out = infer_model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            temperature=1.0,
            repetition_penalty=1.1,
            pad_token_id=infer_tokenizer.eos_token_id,
        )

    reponse = infer_tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()
    return reponse

# ── Tests par catégorie ───────────────────────────────────────
tests = [
    # (commentaire, owner, topology, vendors, cause_attendue)
    ("Grid outage. Grid only site. Awaiting grid return.",
     "IHS", "grid only", "", "Coupure ENEO & Baisse de tension"),

    ("GE fail to start. Grid outage. Generator overload.",
     "IHS", "Good Grid + GE", "", "Defaut GE & Power Cabinet"),

    ("Grid outage. Gen fail to start.",
     "CAMUSAT", "Good Grid + GE", "", "AKTIVCO Defaut GE & Power Cabinet"),

    ("Grid outage. Grid only. No space MDG.",
     "CAMUSAT", "grid only", "", "AKTIVCO Coupure ENEO & Baisse de tension"),

    ("Site MTN impacted. Sharing issue reported.",
     "MTN", "", "", "Sharing"),

    ("Travaux OCM en cours. Démontage antenne prévu.",
     "CAMUSAT", "", "HUAWEI", "Projet OCM (HUAWEI)"),

    ("RRU défaillant. BSS hardware issue confirmé.",
     "", "", "", "BSS Hardware issue"),

    ("AOF fiber cut confirmed. Backbone down.",
     "", "", "", "Fiber AOF"),

    ("Access issue. Landlord blocked access.",
     "", "", "", "ACCESS-ISSUE"),

    ("MPR link down. Microwave path issue.",
     "", "", "", "MPR issue"),
]

print("🧪 TESTS DU MODÈLE FINAL")
print("=" * 60)

corrects = 0
for commentaire, owner, topology, vendors, attendu in tests:
    reponse = predire(commentaire, owner, topology, vendors)

    # Extraire la cause de la réponse
    cause = ""
    for ligne in reponse.split("\n"):
        if "Cause :" in ligne or "Cause:" in ligne:
            cause = ligne.split("Cause :")[-1].split("Cause:")[-1].strip()
            break

    ok = attendu.lower() in cause.lower() or cause.lower() in attendu.lower()
    corrects += ok

    print(f"\n  Commentaire  : {commentaire[:60]}...")
    print(f"  Attendu      : {attendu}")
    print(f"  Prédit       : {cause}")
    print(f"  Résultat     : {'✅' if ok else '❌'}")

print(f"\n{'='*60}")
print(f"  Score : {corrects}/{len(tests)} ({corrects/len(tests)*100:.0f}%)")
print(f"{'='*60}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELLULE UNIQUE — TEST ET ÉVALUATION COMPLÈTE
# Fichier test       : test2.xlsx        (2500 lignes)
# Fichier validation : validation2.xlsx  (CAUSE, Verification, Codesite)
# Sortie             : resultats_test2.xlsx + rapport_performance.xlsx
# ══════════════════════════════════════════════════════════════

import os, re, time, json, warnings
import pandas as pd
from collections import Counter, defaultdict
from google.colab import drive, files

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
# ÉTAPE 0 — CONFIGURATION
# ─────────────────────────────────────────────────────────────
drive.mount("/content/drive", force_remount=True)

BASE            = "/content/drive/MyDrive/Lynes"
CHEMIN_TEST     = f"{BASE}/test2.xlsx"
CHEMIN_VALID    = f"{BASE}/validation2.xlsx"
CHEMIN_SORTIE   = f"{BASE}/resultats_test2.xlsx"
CHEMIN_RAPPORT  = f"{BASE}/rapport_performance.xlsx"
CHEMIN_LORA     = f"{BASE}/finetuned_lora_v2"   # ← adapte si nécessaire
CHEMIN_MERGE    = f"{BASE}/model_final_v2"       # ← adapte si nécessaire

# Catégories entraînées — évaluation uniquement sur celles-ci
CATEGORIES_ENTRAINEMENT = {
    "Coupure ENEO & Baisse de tension",
    "AKTIVCO Coupure ENEO & Baisse de tension",
    "Defaut GE & Power Cabinet",
    "AKTIVCO Defaut GE & Power Cabinet",
    "Sharing",
    "Sites strategiques & DataCenter",
    "Sites strategiques, MLL, Pylone, etc.",
    "Projet OCM (HUAWEI)",
    "Projet OCM (ZTE, NOKIA, autres projets)",
    "BSS Hardware issue",
    "ACCESS-ISSUE",
    "MPR issue",
    "ODU HS",
    "IP & VLAN",
    "Fiber AOF",
    "Fiber CAMTEL",
    "SPARE-ISSUE",
    "SPARE-HS",
    "SAT",
    "Exclu",
    "Warehouse HUAWEI",
}

# Mots-clés détectant un site impacté (insensible à la casse)
PATTERNS_IMPACT = [
    r"indisponibilit[eé]\s+des\s+services?\s+voix\s*[&et]+\s*data\s+dans\s+la\s+localit[eé]\s+de\s+(.+)",
    r"impacted?\s+by\s+(.+)",
    r"impact[eé]r?\s+par\s+(.+)",
    r"incident\s+at\s+(.+)",
    r"probabl[e]?\s+probl[eè]me\s+de\s+.{0,40}\s+sur\s+le\s+site\s+de\s+(.+)",
    r"affect(?:ed)?\s+by\s+.{0,40}\s+at\s+(.+)",
]
REGEX_IMPACT = [re.compile(p, re.IGNORECASE) for p in PATTERNS_IMPACT]

# Pattern code site (ex : LIT_123, CTR_456, EST_070)
REGEX_CODE_SITE = re.compile(r"\b([A-Z]{2,4}_\d{3,4})\b", re.IGNORECASE)

print("✅ Configuration OK")
print(f"   Test       : {CHEMIN_TEST}")
print(f"   Validation : {CHEMIN_VALID}")

# ─────────────────────────────────────────────────────────────
# ÉTAPE 1 — CHARGEMENT DU MODÈLE
# ─────────────────────────────────────────────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print("\n" + "="*60)
print("🤖 CHARGEMENT DU MODÈLE")
print("="*60)

# Essayer d'abord le modèle mergé, sinon le modèle LoRA
CHEMIN_MODELE = CHEMIN_MERGE if os.path.exists(CHEMIN_MERGE) else CHEMIN_LORA
print(f"   Chargement depuis : {CHEMIN_MODELE}")

infer_tokenizer = AutoTokenizer.from_pretrained(CHEMIN_MODELE, trust_remote_code=True)
infer_tokenizer.pad_token    = infer_tokenizer.eos_token
infer_tokenizer.padding_side = "left"

infer_model = AutoModelForCausalLM.from_pretrained(
    CHEMIN_MODELE,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
infer_model.eval()

# Lire le system prompt depuis le dataset
with open(f"{BASE}/dataset/final/train.jsonl") as f:
    SYSTEM_PROMPT = json.loads(f.readline())["messages"][0]["content"]

print(f"✅ Modèle chargé")
print(f"   VRAM utilisée : {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.mem_get_info()[0])/1024**3:.1f} GiB")

# ─────────────────────────────────────────────────────────────
# ÉTAPE 2 — FONCTIONS UTILITAIRES
# ─────────────────────────────────────────────────────────────

def propre(v):
    """Nettoie une valeur pandas."""
    s = str(v).strip()
    return "" if s.lower() in ["nan", "n/a", "none", "", "-"] else s

def est_impacte(commentaire):
    """Retourne True si le commentaire indique un site impacté."""
    for regex in REGEX_IMPACT:
        if regex.search(commentaire):
            return True
    return False

def extraire_site_impactant(commentaire):
    """
    Extrait le code site et/ou le nom du site impactant.
    Retourne (code_site, texte_brut_apres_motcle).
    """
    for regex in REGEX_IMPACT:
        m = regex.search(commentaire)
        if m:
            texte_apres = m.group(1).strip()
            # Chercher un code site dans le texte après le mot-clé
            codes = REGEX_CODE_SITE.findall(texte_apres)
            code  = codes[0].upper() if codes else ""
            # Nom du site = le reste sans le code site
            nom   = REGEX_CODE_SITE.sub("", texte_apres).strip(" ,;.-").split("\n")[0][:80]
            return code, nom
    return "", ""

def predire(commentaire, owner="", topology="", vendors=""):
    """
    Lance l'inférence et retourne (analyse, justification, cause).
    """
    user = f'Commentaire : "{commentaire.strip()}"'
    if propre(owner):    user += f"\nOwner : {propre(owner)}"
    if propre(topology): user += f"\nTopology : {propre(topology)}"
    if propre(vendors):  user += f"\nVendors : {propre(vendors)}"

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user},
    ]
    prompt = infer_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = infer_tokenizer(prompt, return_tensors="pt").to(infer_model.device)

    with torch.no_grad():
        out = infer_model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False,
            temperature=1.0,
            repetition_penalty=1.1,
            pad_token_id=infer_tokenizer.eos_token_id,
        )

    reponse = infer_tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()

    # Extraire les 3 parties
    analyse      = ""
    justification = ""
    cause        = ""
    for ligne in reponse.split("\n"):
        if ligne.startswith("Analyse :"):
            analyse = ligne.replace("Analyse :", "").strip()
        elif ligne.startswith("Justification :"):
            justification = ligne.replace("Justification :", "").strip()
        elif ligne.startswith("Cause :"):
            cause = ligne.replace("Cause :", "").strip()

    # Si extraction échoue, récupérer la dernière ligne non vide
    if not cause:
        lignes_non_vides = [l.strip() for l in reponse.split("\n") if l.strip()]
        if lignes_non_vides:
            cause = lignes_non_vides[-1]

    return analyse, justification, cause

# ─────────────────────────────────────────────────────────────
# ÉTAPE 3 — LECTURE DU FICHIER TEST
# ─────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("📂 LECTURE DU FICHIER TEST")
print("="*60)

df = pd.read_excel(CHEMIN_TEST, dtype=str)
print(f"   Lignes totales : {len(df)}")
print(f"   Colonnes       : {list(df.columns)}")

# Détecter les colonnes contextuelles
col_com = "Verifications"
col_codesite = "Codesite"
col_own = next((c for c in df.columns if c.lower() == "owner"), None)
col_top = next((c for c in df.columns if "topolog" in c.lower()), None)
col_ven = next((c for c in df.columns
                if "vendor" in c.lower() or "equipement" in c.lower()), None)

print(f"   Colonne commentaire : {col_com}")
print(f"   Colonne code site   : {col_codesite}")
print(f"   Colonne Owner       : {col_own}")
print(f"   Colonne Topology    : {col_top}")
print(f"   Colonne Vendors     : {col_ven}")

# Initialiser les colonnes de sortie
df["TYPE_CAS"]       = ""
df["SITE_IMPACTANT"] = ""
df["ANALYSE"]        = ""
df["JUSTIFICATION"]  = ""
df["CAUSE_PREDITE"]  = ""
df["TEMPS_S"]        = 0.0

# ─────────────────────────────────────────────────────────────
# ÉTAPE 4 — CLASSIFICATION DES LIGNES
# ─────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("🔍 CLASSIFICATION DES LIGNES")
print("="*60)

idx_no_root  = []
idx_normaux  = []
idx_impactes = []

for idx, row in df.iterrows():
    com = propre(str(row.get(col_com, "")))
    if not com:
        df.at[idx, "TYPE_CAS"]      = "NO_ROOT_CAUSE"
        df.at[idx, "CAUSE_PREDITE"] = "no root cause"
        df.at[idx, "ANALYSE"]       = "Aucun commentaire disponible."
        df.at[idx, "JUSTIFICATION"] = "Pas d'information sur l'incident."
        idx_no_root.append(idx)
    elif est_impacte(com):
        df.at[idx, "TYPE_CAS"] = "IMPACTE"
        idx_impactes.append(idx)
    else:
        df.at[idx, "TYPE_CAS"] = "NORMAL"
        idx_normaux.append(idx)

print(f"   No root cause : {len(idx_no_root)}")
print(f"   Cas normaux   : {len(idx_normaux)}")
print(f"   Cas impactés  : {len(idx_impactes)}")

# ─────────────────────────────────────────────────────────────
# ÉTAPE 5 — TRAITEMENT 1 : CAS NORMAUX
# ─────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("🤖 TRAITEMENT 1 — CAS NORMAUX")
print("="*60)

# Dictionnaire codesite → infos pour résolution des cas impactés
dict_resultats = {}   # codesite_upper → {cause, analyse, justification}
dict_nom_site  = {}   # nom_site_lower → codesite_upper (fallback)

t_debut = time.time()

for i, idx in enumerate(idx_normaux, 1):
    row = df.loc[idx]
    com = propre(str(row.get(col_com, "")))
    own = propre(str(row.get(col_own, ""))) if col_own else ""
    top = propre(str(row.get(col_top, ""))) if col_top else ""
    ven = propre(str(row.get(col_ven, ""))) if col_ven else ""

    t0 = time.time()
    try:
        analyse, justification, cause = predire(com, own, top, ven)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        analyse, justification, cause = "", "", "ERREUR_OOM"
    except Exception as e:
        analyse, justification, cause = "", "", f"ERREUR : {str(e)[:60]}"

    df.at[idx, "ANALYSE"]       = analyse
    df.at[idx, "JUSTIFICATION"] = justification
    df.at[idx, "CAUSE_PREDITE"] = cause
    df.at[idx, "TEMPS_S"]       = round(time.time() - t0, 2)

    # Indexer par codesite pour les cas impactés
    codesite = propre(str(row.get(col_codesite, ""))).upper()
    if codesite and cause not in ["ERREUR_OOM", ""] and not cause.startswith("ERREUR"):
        dict_resultats[codesite] = {
            "cause":         cause,
            "analyse":       analyse,
            "justification": justification,
        }

    # Afficher progression
    if i % 50 == 0 or i == 1:
        elapsed = time.time() - t_debut
        vitesse = i / elapsed if elapsed > 0 else 0
        restant = (len(idx_normaux) - i) / vitesse if vitesse > 0 else 0
        print(f"   {i:>4}/{len(idx_normaux)} | {vitesse:.1f} lignes/s | ETA {restant/60:.1f} min")

print(f"\n✅ {len(idx_normaux)} cas normaux traités")
print(f"   Sites indexés pour résolution : {len(dict_resultats)}")

# ─────────────────────────────────────────────────────────────
# ÉTAPE 6 — TRAITEMENT 2 : CAS IMPACTÉS
# ─────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("🔗 TRAITEMENT 2 — CAS IMPACTÉS")
print("="*60)

resolus     = 0
non_resolus = 0

for idx in idx_impactes:
    row = df.loc[idx]
    com = propre(str(row.get(col_com, "")))

    code_impactant, nom_impactant = extraire_site_impactant(com)
    df.at[idx, "SITE_IMPACTANT"] = code_impactant or nom_impactant or "non trouvé"

    # Chercher d'abord par code site
    info = None
    if code_impactant and code_impactant.upper() in dict_resultats:
        info = dict_resultats[code_impactant.upper()]

    # Sinon chercher par nom de site (recherche partielle)
    if not info and nom_impactant:
        for codesite, data in dict_resultats.items():
            # Chercher si le nom apparaît dans le codesite ou vice-versa
            if nom_impactant.lower()[:8] in codesite.lower():
                info = data
                break

    if info:
        df.at[idx, "CAUSE_PREDITE"] = info["cause"]
        df.at[idx, "ANALYSE"] = (
            f"Site impacté par {code_impactant or nom_impactant}. "
            f"Cause héritée du site impactant. "
            f"{info['analyse'][:100]}"
        )
        df.at[idx, "JUSTIFICATION"] = (
            f"Ce site est impacté par {code_impactant or nom_impactant} "
            f"dont la cause est : {info['cause']}"
        )
        resolus += 1
    else:
        df.at[idx, "CAUSE_PREDITE"] = "SITE_IMPACTANT_NON_RESOLU"
        df.at[idx, "ANALYSE"] = (
            f"Site impacté détecté. "
            f"Site impactant : {code_impactant or nom_impactant or 'non identifié'}. "
            f"Le site source n'a pas été trouvé dans les résultats."
        )
        df.at[idx, "JUSTIFICATION"] = "Site impactant absent du fichier de traitement."
        non_resolus += 1

print(f"   ✅ Résolus     : {resolus}")
print(f"   ⚠️  Non résolus : {non_resolus}")

# ─────────────────────────────────────────────────────────────
# ÉTAPE 7 — FUSION AVEC VALIDATION2
# ─────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("📊 FUSION AVEC VALIDATION2")
print("="*60)

df_valid = pd.read_excel(CHEMIN_VALID, dtype=str)
print(f"   Lignes validation : {len(df_valid)}")
print(f"   Colonnes          : {list(df_valid.columns)}")

# Normaliser les codesites pour la fusion
df["Codesite_upper"]       = df[col_codesite].str.strip().str.upper()
df_valid["Codesite_upper"] = df_valid["Codesite"].str.strip().str.upper()

# Fusion par Codesite
df_fusion = df.merge(
    df_valid[["Codesite_upper", "CAUSE"]].rename(columns={"CAUSE": "CAUSE_REELLE"}),
    on="Codesite_upper",
    how="left",
)
df_fusion["CAUSE_REELLE"] = df_fusion["CAUSE_REELLE"].fillna("").str.strip()
df_fusion["CAUSE_PREDITE"] = df_fusion["CAUSE_PREDITE"].fillna("").str.strip()

# Colonne résultat ✓ / ✗
def calculer_resultat(row):
    if row["TYPE_CAS"] == "NO_ROOT_CAUSE":
        return "—"
    if not row["CAUSE_REELLE"]:
        return "—"
    if row["CAUSE_PREDITE"].lower() == row["CAUSE_REELLE"].lower():
        return "✓"
    return "✗"

df_fusion["RESULTAT"] = df_fusion.apply(calculer_resultat, axis=1)

print(f"   Lignes fusionnées : {len(df_fusion)}")

# ─────────────────────────────────────────────────────────────
# ÉTAPE 8 — CALCUL DES MÉTRIQUES
# ─────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("📈 CALCUL DES MÉTRIQUES")
print("="*60)

# Filtrer : uniquement les catégories entraînées + avec vraie cause
df_eval = df_fusion[
    (df_fusion["CAUSE_REELLE"].isin(CATEGORIES_ENTRAINEMENT)) &
    (df_fusion["TYPE_CAS"] != "NO_ROOT_CAUSE")
].copy()

total   = len(df_eval)
correct = (df_eval["RESULTAT"] == "✓").sum()
acc     = correct / total * 100 if total > 0 else 0

print(f"   Total évalué  : {total}")
print(f"   ✓ Corrects    : {correct} ({acc:.1f}%)")
print(f"   ✗ Incorrects  : {total - correct} ({100-acc:.1f}%)")

# Métriques par catégorie
stats_cat = defaultdict(lambda: {"correct": 0, "total": 0, "erreurs": []})
for _, row in df_eval.iterrows():
    cat = row["CAUSE_REELLE"]
    stats_cat[cat]["total"] += 1
    if row["RESULTAT"] == "✓":
        stats_cat[cat]["correct"] += 1
    else:
        stats_cat[cat]["erreurs"].append(row["CAUSE_PREDITE"])

print(f"\n   {'Catégorie':<45} {'OK':>4} {'Tot':>5} {'Acc%':>7}")
print("   " + "-"*65)
for cat in sorted(stats_cat.keys()):
    s   = stats_cat[cat]
    a   = s["correct"] / s["total"] * 100 if s["total"] > 0 else 0
    flg = "✅" if a >= 80 else ("⚠️" if a >= 50 else "❌")
    print(f"   {flg} {cat:<43} {s['correct']:>4} {s['total']:>5} {a:>6.1f}%")
print("   " + "-"*65)
print(f"   {'TOTAL':<45} {correct:>4} {total:>5} {acc:>6.1f}%")

# ─────────────────────────────────────────────────────────────
# ÉTAPE 9 — SAUVEGARDE FICHIER RÉSULTATS
# ─────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("💾 SAUVEGARDE FICHIER RÉSULTATS")
print("="*60)

# Colonnes finales à exporter
cols_export = [
    col_codesite, col_com,
    "TYPE_CAS", "SITE_IMPACTANT",
    "ANALYSE", "JUSTIFICATION",
    "CAUSE_PREDITE", "CAUSE_REELLE",
    "RESULTAT", "TEMPS_S",
]
# Ajouter colonnes contextuelles si présentes
for c in [col_own, col_top, col_ven]:
    if c and c not in cols_export:
        cols_export.append(c)

cols_export = [c for c in cols_export if c and c in df_fusion.columns]

with pd.ExcelWriter(CHEMIN_SORTIE, engine="openpyxl") as writer:
    # Feuille 1 : tous les résultats
    df_fusion[cols_export].to_excel(writer, sheet_name="Résultats", index=False)

    # Feuille 2 : résumé par catégorie
    resume_data = []
    for cat in sorted(stats_cat.keys()):
        s   = stats_cat[cat]
        a   = s["correct"] / s["total"] * 100 if s["total"] > 0 else 0
        top = Counter(s["erreurs"]).most_common(1)
        resume_data.append({
            "Catégorie"       : cat,
            "✓ Corrects"      : s["correct"],
            "Total"           : s["total"],
            "✗ Incorrects"    : s["total"] - s["correct"],
            "Accuracy (%)"    : round(a, 1),
            "Statut"          : "✅ Bon" if a >= 80 else ("⚠️ Moyen" if a >= 50 else "❌ Faible"),
            "Erreur principale": top[0][0] if top else "—",
        })
    pd.DataFrame(resume_data).to_excel(writer, sheet_name="Par catégorie", index=False)

    # Feuille 3 : résumé global
    stats_type = df_fusion["TYPE_CAS"].value_counts().to_dict()
    global_data = {
        "Indicateur": [
            "Total lignes fichier test",
            "Sans commentaire (no root cause)",
            "Cas normaux traités",
            "Cas impactés résolus",
            "Cas impactés non résolus",
            "──────────────────────────",
            "Total évalué (catégories entraînées)",
            "✓ Prédictions correctes",
            "✗ Prédictions incorrectes",
            "Accuracy globale (%)",
            "──────────────────────────",
            "Catégories ≥ 80% accuracy",
            "Catégories 50-79% accuracy",
            "Catégories < 50% accuracy",
        ],
        "Valeur": [
            len(df),
            len(idx_no_root),
            len(idx_normaux),
            resolus,
            non_resolus,
            "──────────────────────────",
            total,
            correct,
            total - correct,
            f"{acc:.1f}%",
            "──────────────────────────",
            sum(1 for s in stats_cat.values()
                if s["total"] > 0 and s["correct"]/s["total"] >= 0.8),
            sum(1 for s in stats_cat.values()
                if s["total"] > 0 and 0.5 <= s["correct"]/s["total"] < 0.8),
            sum(1 for s in stats_cat.values()
                if s["total"] > 0 and s["correct"]/s["total"] < 0.5),
        ]
    }
    pd.DataFrame(global_data).to_excel(writer, sheet_name="Résumé global", index=False)

    # Feuille 4 : confusions détaillées
    confusion_rows = []
    for cat, s in sorted(stats_cat.items()):
        if s["erreurs"]:
            for err, nb in Counter(s["erreurs"]).most_common(5):
                confusion_rows.append({
                    "Cause réelle"   : cat,
                    "Cause prédite"  : err,
                    "Nb occurrences" : nb,
                    "Impact (%)"     : round(nb / s["total"] * 100, 1),
                })
    if confusion_rows:
        pd.DataFrame(confusion_rows).to_excel(writer, sheet_name="Confusions", index=False)

print(f"✅ Résultats sauvegardés : {CHEMIN_SORTIE}")

# ─────────────────────────────────────────────────────────────
# ÉTAPE 10 — RAPPORT DE PERFORMANCE POUR CHEF
# ─────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("📋 RAPPORT DE PERFORMANCE — PRÉSENTATION CHEF")
print("="*60)

print(f"""
┌─────────────────────────────────────────────────────────┐
│           RAPPORT DE PERFORMANCE — MODÈLE NOC            │
├─────────────────────────────────────────────────────────┤
│  Fichier analysé    : test2.xlsx ({len(df)} lignes)
│  Sans commentaire   : {len(idx_no_root)} → 'no root cause' (exclus)
│  Traités modèle     : {len(idx_normaux)} cas normaux
│  Cas impactés       : {len(idx_impactes)} ({resolus} résolus, {non_resolus} non résolus)
├─────────────────────────────────────────────────────────┤
│  PERFORMANCE SUR CATÉGORIES ENTRAÎNÉES
│  Total évalué       : {total}
│  ✓ Corrects         : {correct} ({acc:.1f}%)
│  ✗ Incorrects       : {total - correct} ({100-acc:.1f}%)
├─────────────────────────────────────────────────────────┤
│  CATÉGORIES
│  ✅ Accuracy ≥ 80%  : {sum(1 for s in stats_cat.values() if s['total']>0 and s['correct']/s['total']>=0.8)} catégories
│  ⚠️  Accuracy 50-79% : {sum(1 for s in stats_cat.values() if s['total']>0 and 0.5<=s['correct']/s['total']<0.8)} catégories
│  ❌ Accuracy < 50%  : {sum(1 for s in stats_cat.values() if s['total']>0 and s['correct']/s['total']<0.5)} catégories
└─────────────────────────────────────────────────────────┘
""")

# ─────────────────────────────────────────────────────────────
# ÉTAPE 11 — TÉLÉCHARGEMENT EN LOCAL
# ─────────────────────────────────────────────────────────────
print("="*60)
print("⬇️  TÉLÉCHARGEMENT DES FICHIERS EN LOCAL")
print("="*60)

print("   Téléchargement : resultats_test2.xlsx ...")
files.download(CHEMIN_SORTIE)

print("\n✅ TOUT EST TERMINÉ")
print("   Fichier téléchargé sur ta machine :")
print("   → resultats_test2.xlsx")
print("     Feuilles : Résultats | Par catégorie | Résumé global | Confusions")